# UI Workflow Walkthrough
Manual inspection companion for the Streamlit Data Quality Monitoring UI. Use this notebook to inspect UI-generated config, latest run artifacts, parsed report sheets, context records, and approval publishing inputs.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'config').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from dq_agent import ui_support as ui
ROOT


## Load UI Inputs

In [ ]:
inputs = ui.load_runtime_inputs(ROOT)
display(inputs['table_mappings'])
display(inputs['column_mappings'])
display(inputs['filter_rows'])
display(inputs['context_hints'])
display(inputs['human_tests'])


## Generate Run-Specific UI Config
This writes isolated UI inputs under `logs/<run_id>/ui_inputs/` and does not overwrite canonical files under `inputs/`.

In [ ]:
RUN_ID = ui.utc_run_id('notebook_ui_config')
files = ui.prepare_ui_run(
    ROOT, RUN_ID,
    inputs['table_mappings'], inputs['column_mappings'], inputs['filter_rows'],
    inputs['human_tests'], inputs['context_hints'], inputs['llm'],
)
files


## Latest Run Snapshot

In [ ]:
snapshot = ui.load_monitoring_snapshot(ROOT)
print('Demo data:', snapshot['is_demo'])
display(snapshot['summary'])
display(ui.issue_counts(snapshot))
display(ui.rca_completion(snapshot))


## Report Sheets And Events

In [ ]:
run_id = ui.latest_run_id(ROOT)
print('Latest real run:', run_id)
if run_id:
    report = ui.load_report_workbook(ROOT, run_id)
    print(sorted(report))
    display(ui.consolidated_results(report).head(20))
    display(pd.DataFrame(ui.run_progress(ROOT, run_id)['events']).tail(20))
else:
    print('No real run artifacts found yet.')


## Context And Approvals

In [ ]:
context_records = ui.load_context_records(ROOT)
display(context_records[['record_id', 'context_type', 'subject_key', 'pair_id', 'origin', 'confidence']].head(20))
pending = ui.list_approval_workbooks(ROOT, 'pending')
display([path.name for path in pending])
if pending:
    display(ui.pending_approval_frame(pending[0]).head(20))
